# Neural Networks - Activation Functions

All about:
- Geometry
- Gradient flow
- Expressivity
- Optimisation dynamics

## 1. Why Neural Networks Need Nonlinearity

## 2. Mathematical Properties of Activation Functions

> If a new activation function is invented, how to decide whether it is good?

Criteria:

1. Is it non-linear?
2. Is it differentiable?
3. Does its derivative vanish?
4. Does it saturate?
5. Is it zero-centred?
6. Is it computationally efficient?
7. Is it smooth?
8. Does it preserve variance?
9. Is it numerical stable?
10. Does it help optimisation?

### Non-linearity

```math
\sigma(ax + by) \ne a\sigma(x) + b\sigma(y)
```

### Differentiability

```math
\frac{\partial L}{\partial z} = \frac{\partial L}{\partial a}. \frac{\partial a}{\partial z}
```

### Saturation

An activation is saturated when its derivative approaches zero over large regions.

```math
\lim_{z \rightarrow \infin}\sigma'(z) = 0
```

then the gradient passed backward becomes tiny.

> Vanishing gradient problem.

### Zero-centred outputs

If activation > 0 -> layer receives positive bias -> gradient updates tend to be correlated in one direction.

Else, if outputs are centred around zero, optimisation often proceeds more smoothly because positive and negative corrections can balance each other.

That's why *tanh* is often better than *sigmoid*.

### Smoothness

- Derivatives change continuously.
- ReLU and Softplus

### Monotonicity

Monotonicity means:

```math
z_1 < z_2 \Leftrightarrow \sigma(z_1) \le \sigma(z_2)
```

Some activations like Mish and Swish are non-monotonic and sometimes learn richer representations.

### Computational cost

Large language models perform trillions of activation evaluations. A tiny saving per activation scales to enormous savings.

### Numerical stability

- Avoid large positive/negative inputs, or
- Use numerically stable reformulations.

In [1]:
import numpy as np

# central difference
def numerical_derivative(f, z, eps=1e-6):
    return (f(z + eps) - f(z - eps)) / (2 * eps)

def analyse_activation(f, name):
    z = np.linspace(-5, 5, 1000)
    y = f(z)
    dy = numerical_derivative(f, z)

    print(name)
    print("Output range:", np.min(y), np.max(y))
    print("Derivative range:", np.min(dy), np.max(dy))

In [2]:
def square(z):
    return z**2

analyse_activation(square, "Square")

Square
Output range: 2.5050075100125133e-05 25.0
Derivative range: -10.00000000139778 10.00000000139778


In [3]:
z = np.linspace(-20, 20, 1000)
dy = numerical_derivative(np.tanh, z)

print(dy.min(), dy.max())

0.0 0.9995993058703467


### Measuring computation cost

For large arrays, compare runtimes.

In [4]:
import time

z = np.random.randn(10_000_000)

start = time.time()
np.maximum(0, z)
print("ReLU:", time.time() - start)

start = time.time()
1 / (1 + np.exp(-z))
print("Sigmoid:", time.time() - start)

ReLU: 0.025423049926757812
Sigmoid: 0.12285304069519043


| Property                           | Sigmoid    | tanh     | ReLU                     |
| ---------------------------------- | ---------- | -------- | ------------------------ |
| Nonlinear                          | ✓          | ✓        | ✓                        |
| Differentiable (almost everywhere) | ✓          | ✓        | ✓ (except at 0)          |
| Saturates                          | ✓          | ✓        | Only for negative inputs |
| Zero-centred                       | ✗          | ✓        | ✗                        |
| Smooth                             | ✓          | ✓        | ✗                        |
| Monotonic                          | ✓          | ✓        | ✓                        |
| Cheap                              | Moderate   | Moderate | Excellent                |
| Numerically stable by default      | Needs care | Good     | Excellent                |


## 3. Sigmoid: The First Neural Activation

> Why Sigmoid is used in early networks, then replaced in most deep architectures?

We need a function that:
- maps every real numbers to a probability
- is smooth
- is differentiable everywhere
- behaves like a soft threshold

> **Threshold**: a trigger point determinees whether a neuron fires to pass information to the next layer.

### Desired properties

- Large negative input -> 0
- Large positive input -> 1
- Middle               -> 0.5


```math
\sigma(z) = \frac{1}{1 + e^{-z}}
```

### Sigmoid derivative

```math
\sigma'(z) = \sigma(z)(1 - \sigma(z))
```

> When is learning fastest?

The maximum is $\sigma(z) = 0.5$, at $z = 0$, where the steepest slope is $\frac{1}{2}(1 - \frac{1}{2}) = \frac{1}{4}$.

Therefore,

```math
0 < \sigma'(z) \le 0.25
```

### Why they struggle with Sigmoid

Because its range is 0 -> 0.25, the gradient onlyu shrink as it propagates backwards.

For instance, after 10 layers, even if a deriative cloase to the maximum:

```math
0.25^10 \approx 9.5 \times 10^{-7}
```

Imagine smaller derivative could even shrink more.

### How Sigmoid is used today

Sigmoid is still used in:
- Binary classification
- Logistic regression
- Gating mechanisms (LSTM, GRU where output is between 0 and 1)

## 4. Hyperbolic Tangent (tanh)

> Sigmoid works but training is slow.
> Every sigmoid output is positive.

Gradient descent:
- If activations are centred around zero, the optimiser can make balanced positive and negative corrections.
- If activations are always positive -> correlated updates -> inefficient zig-zag optimisation paths.

### Tanh properties

```math
\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}
```

- range = (-1, 1)
- Relationship to Sigmoid

```math
\tanh(z) = 2\sigma(2z) - 1
```

- Scales and shifts Sigmoid.

Derivative

```math
\tanh'(z) = 1 - \tanh^2(z)
```

### Tanh derivative

```math
0 < \tanh'(z) \le 1
```

Max at $z = 0$, where $\tanh(0) = 0$ and $\tanh'(z) = 1$

- tanh preserves gradients much better near the origin.

In [6]:
import numpy as np

class Tanh:
    def forward(self, z):
        ez = np.exp(z)
        emz = np.exp(-z)
        return (ez - emz) / (ez + emz)

    def backward(self, z):
        t = self.forward(z)
        return 1 - t**2

In [7]:
tanh = Tanh()
z = np.linspace(-5,5,100)
my = tanh.forward(z)
numpy = np.tanh(z)

print(np.max(np.abs(my-numpy)))

2.220446049250313e-16


In [8]:
# Numerical derivative

eps = 1e-6

num = (
    tanh.forward(z+eps)
    -
    tanh.forward(z-eps)
)/(2*eps)

ana = tanh.backward(z)

print(np.max(np.abs(num-ana)))

1.6094658938925477e-10


In [9]:
# Compare output ranges

sigmoid = lambda z: 1/(1+np.exp(-z))

z = np.linspace(-5,5,11)

print(sigmoid(z))
print(np.tanh(z))

[0.00669285 0.01798621 0.04742587 0.11920292 0.26894142 0.5
 0.73105858 0.88079708 0.95257413 0.98201379 0.99330715]
[-0.9999092  -0.9993293  -0.99505475 -0.96402758 -0.76159416  0.
  0.76159416  0.96402758  0.99505475  0.9993293   0.9999092 ]


### Why tanh is still not good

- Saturation
- But still it is used in:
  - RNN (hidden state in LSTM, GRU)
  - physics-informed NN
  - implicit neural representations
  - problems where target in [-1, 1]
- Representations produced by tanh more balanced than Sigmoid, which in turn makes optimisation easier

## 5. ReLU: The Deep Learning Revolution

> A simpler function than Sigmoid, but outperforms.

```math
\text{ReLU}(z) - \max(0, z)
```

- Both Sigmoid and tanh compress input to [0, 1] and [-1, 1] -> data loss.
- ReLU preserves positive info.

```math
ReLU'(z) = \begin{cases}
    0, & z < 0 \\
    1, & z > 0
\end{cases}
```

- Derivative is undefinded at $z = 0$. -> define it = 0 (like in PyTorch) or 1.

Recall

```math
\delta^{(l)} = (W^{(l+1)})^T\delta^{(l+1)} \odot \sigma'(z^{(l)})
```

Since $\sigma'(z^{(l)}) = 1$,

```math
\delta^{(l)} = (W^{(l+1)})^T\delta^{(l+1)}
```

- No additional shrink.
- Reduces vanishing gradient.
- Sparse activation (negative -> 0).

In [10]:
import numpy as np

class ReLU:

    def forward(self, z):
        return np.maximum(0, z)

    def backward(self, z):
        return (z > 0).astype(float)

In [13]:
eps = 1e-6

relu = ReLU()
z = np.linspace(-3,3,1000)

num = (
    relu.forward(z+eps)
    -
    relu.forward(z-eps)
)/(2*eps)

ana = relu.backward(z)


| Property       | Sigmoid  | tanh     | ReLU                  |
| -------------- | -------- | -------- | --------------------- |
| Nonlinear      | ✅        | ✅        | ✅                     |
| Smooth         | ✅        | ✅        | ❌                     |
| Zero-centred   | ❌        | ✅        | ❌                     |
| Saturates      | ✅        | ✅        | Only on negative side |
| Cheap          | Moderate | Moderate | ⭐ Excellent           |
| Max derivative | 0.25     | 1        | 1                     |
| Sparse outputs | ❌        | ❌        | ✅                     |


### Why ReLU is not globally linear

- two regions: z < 0, z >= 0
- Piecewise linear regions.
- Treated as a gate: block or keep.

### Why ReLU is not perfect

- Always negative input -> ReLU = 0 -> ReLU' = 0 -> **dying ReLU problem**.

## 6. Variants of ReLU

### Leaky ReLU

Instead of blocking entirely negative values, make a narrow dirt path for it to slightly go through.

```math
f(z) = \begin{cases}
    z, & z > 0 \\
    \alpha z, & z \le 0
\end{cases}
```

where $0 < \alpha \ll 1$. They normally choose $\alpha = 0.01$.

In [14]:
import numpy as np

class LeakyReLU:

    def __init__(self, alpha=.01):
        self.alpha = alpha

    def forward(self, z):
        return np.where(z > 0, z, self.alpha * z)
    
    def backward(self, z):
        return np.where(z > 0, 1.0, self.alpha)

### Parametric ReLU (PReLU)

Instead of fixing $\alpha = 0.01$ -> trainable param.

```math
\frac{\partial L}{\partial \alpha} = z
```

In [15]:
import numpy as np

class PReLU:

    def __init__(self, alpha=.25):
        self.alpha = alpha

    def forward(self, z):
        return np.where(z > 0, z, self.alpha * z)

    def backward(self, z):
        return np.where(z > 0, 1.0, self.alpha)
    
    def grad_a(self, z):
        return np.where(z > 0, 0.0, z)

### Randomised ReLU (RReLU)

```math
\alpha ~ U(l, u)
```

e.g., $\alpha ~ U(0.1, 0.3)$.

- Inference uses the avg value. -> regularisation.
- Inject controlled noise during training to discourage over-reliance on specific activations.

In [16]:
import numpy as np

class RReLU:

    def __init__(self, low=.1, high=.3):
        self.low = low
        self.high = high
        self.alpha = 0.0

    def forward(self, z, training=True):
        if training:
            self.alpha = np.random.uniform(self.low, self.high, size=z.shape)
        else:
            self.alpha = (self.low + self.high)/2
        return np.where(z > 0, z, self.alpha * z)
    
    def backward(self, z):
        return np.where(z > 0, 1.0, self.alpha)

### ReLU6

```math
f(z) = \min(\max(0, z), 6)
```

- First used in MobileNet
- limiting activation range -> low-precision inference (e.g., 8-bit int) more stable and accurate on mobile hardware.

In [17]:
import numpy as np

class ReLU6:

    def forward(self, z):
        return np.minimum(
            np.maximum(0, z),
            6
        )
    
    def backward(self, z):
        return np.where((z > 0) & (z < 6), 1.0, 0.0)

### Comparison

| Property             | ReLU     | Leaky ReLU       | PReLU            | RReLU            | ReLU6    |
| -------------------- | -------- | ---------------- | ---------------- | ---------------- | -------- |
| Dead neurons         | Possible | Much less likely | Much less likely | Much less likely | Possible |
| Trainable parameters | No       | No               | Yes              | No               | No       |
| Randomness           | No       | No               | No               | Yes              | No       |
| Bounded outputs      | No       | No               | No               | No               | Yes      |
| Computational cost   | Very low | Very low         | Low              | Moderate         | Very low |


## 7. Smooth Activations

> ReLU solved vanishing gradient problem, but
> Can we keep ReLU's optimisation advantages while making gradients smoother and information flow richer?

- Smooth ReLU family:
  - ELU
  - SELU
  - Softplus
  - Swish
  - Mish
  - GELU
- From piecewise linear functions -> Smooth non-linear functions.

### Recall ReLU's problems

- Not differentiable (0, undefined at 0, 1) -> Can optimisation become easier with smooth gradients???
- Dead neurons: negative input -> ReLU = 0 -> ReLU' = 0.
- Hidden activations are positive bias (ReLU >= 0).
- Information discarded for negative input.

### ELU (Exponential Linear Unit)

```math
f(x) = \begin{cases}
    x, & x > 0 \\
    \alpha(e^x - 1), & x \le 0
\end{cases}
```

- When $x \le 0$: $f(x)$ approaches $-\alpha$.

**Derivative**

```math
f'(x) = \begin{cases}
    1, & x > 0 \\
    \alpha e^x = f(x) + \alpha, & x \le 0
\end{cases}
```

### SELU (Scaled ELU)

> Can ELU automatically normalise activations?

```math
\text{SELU}(x) = \lambda \begin{cases}
    x, & x > 0 \\
    \alpha(e^x - 1), & x \le 0
\end{cases}
```

- Researchers need layer outputs tend toward mean = 0, variance = 1. So they chose:
  - $\alpha \approx 1.6733$
  - $\lambda \approx 1.0507$
- **Self-normalisation**.

**Derivative**

```math
\text{SELU}'(x) = \begin{cases}
    \lambda, & x > 0 \\
    \lambda\alpha e^x, & x \le 0
\end{cases}
```

### Softplus

Smoothly approximate ReLU.

```math
f(x) = \ln(1 + e^x)
```

- Notice this looks like ReLU:
  - Large positive x -> $e^x$: $\ln(1 + e^x) \approx \ln(e^x) = x$
  - Large negative x -> $e^x \rightarrow 0$: $\ln(1) = 0$

**Derivative**

```math
f'(x) = \frac{1}{1 + e^x}\cdot e^x = \frac{e^x}{1 + e^x} = \frac{1}{1 + e^{-x}} = \sigma(x)
```

> This is beautiful because sigmoid output range is [0, 1] -> Perfect for predicting probabilities.

### Swish

At Google, researchers proposed, with $\sigma(x)$ is the sigmoid function.

```math
f(x) = x\sigma(x)
```

- Positive x -> f(x) slightly reduce
- Negative x -> f(x) shrinks more, but never saturated to 0.

**Derivative**

```math
f'(x) = \sigma + x\sigma' = \sigma + x\sigma(1 - \sigma)
```

- Derivative can exceed 1. -> accerelate optimisation.

### GELU (Gaussian Error Linear Unit)

Used by nearly every modern transformers:
- BERT
- GPT
- PaLM
- Gemini
- Llama
- Claude

```math
f(x) = x\Phi(x)
```

where $\Phi(x)$ is Gaussian CDF (Cumulative Distribution Function).

> Instead of saying "keep everything positive", GELU says "keep inputs with probability determined by a Gaussian."

- Which means
  - Small positives are only partially passed
  - Large positives pass almost fully
  - Negative values are softly suppressed
- Exact GELU is expensive.
- They use approximation in many libraries

```math
0.5x(1 + \tanh(\sqrt{\frac{2}{\pi}}(x + 0.044715x^3)))
```

### Mish

```math
f(x) = x\tanh(\text{Softplus}(x)) = x\tanh(\ln(1 + e^x))
```

- Combine: tanh, softplus, identity
- Very smooth
- Non-monotonic

### Implementation

In [1]:
import numpy as np

class Activation:

    def forward(self, x):
        return NotImplementedError
    
    def backward(self, x):
        return NotImplementedError

In [2]:
class ELU(Activation):

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def forward(self, x):
        return np.where(x > 0, x, self.alpha*(np.exp(x) - 1))
    
    def backward(self, x):
        return np.where(x > 0, 1.0, self.alpha*np.exp(x))

In [5]:
class Softplus(Activation):

    def forward(self, x):
        return np.log1p(np.exp(x))
    
    def backward(self, x):
        return 1 / (1 + np.exp(-x))

In [4]:
class Swish(Activation):
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def forward(self, x):
        return x * self.sigmoid(x)
    
    def backward(self, x):
        return self.sigmoid(x) + x*self.sigmoid(x)(1 - self.sigmoid(x))

In [6]:
class GELUApprox(Activation):

    def forward(self, x):
        return 0.5*x*(1 + np.tanh(
            np.sqrt(2 / np.pi) * (x + 0.044715*x^3)
        ))

In [ ]:
# Remember to do numerical derivative test

# Torch

import torch

torch.nn.ELU
torch.nn.SELU
torch.nn.Softplus
torch.nn.GELU
torch.nn.SiLU   # Swish
torch.nn.Mish

### Summary

- Major design directions:
  - **Exponential smoothing**: ELU and SELU introduce smooth negative outputs, improving optimisation and, in SELU's case, encouraging self-normalisation.
  - **Smooth gating**: Softplus, Swish, GELU, and Mish softly modulate activations instead of making hard on/off decisions. Rather than a binary gate like ReLU, they behave more like a continuous "importance weighting."
- Modern architectures:
  - **CNNs**: ReLU and its variants remain common because they're computationally cheap and work well with Batch Normalisation.
  - **Transformers**: Smooth activations dominate. Nearly all large language models—including GPT-style models, PaLM, Llama, Gemma, Claude, and many others—use GELU or closely related gated variants because they empirically improve optimisation and downstream performance.
  - **MLPs in foundation models**: Recent architectures increasingly use gated MLPs (such as SwiGLU and GeGLU), where Swish- or GELU-like nonlinearities act as learned gates rather than simple pointwise activations. This idea extends the same smooth-gating philosophy even further.

```
Sigmoid
      │
      ├── tanh
      │
      └── ReLU
             │
             ├── Leaky ReLU
             ├── PReLU
             ├── ReLU6
             │
             ├── ELU
             │      └── SELU
             │
             └── Smooth ReLU
                    ├── Softplus
                    ├── Swish (SiLU)
                    ├── GELU
                    └── Mish
```

## 8. Derivatives and Gradient Flow

Isolated activation objects -> learning dynamics of a network.

At each layer $l$:

```math
\delta^{(l)} = (W^{(l+1)})^T\delta^{(l+1)} \odot f'(z^{(l)})
```

So with $A_i = (W_i)^TD_i$, and $D_i = \text{diag}(f'(z_i))$:

```math
\delta_1 = A_1A_2...A_{L-1}A_L\delta_L = (\prod_{l=2}^L W_l^TD_l)\delta_L
```

- Matrix multiplication
- Stable gradient flow = gradient magnitude preserved. Else
  - Vanishing gradient
  - Exploding gradient
- The scale (up or down) of gradient flow depends on matrix $D$, which depends on activation functions.
- Initialisation also matters:
  - **Xavier and He initialisation**: choose variance of the initial weights so that signals and gradients neither explode nor vanish on average.
- **Signals** = information flow in networks:
  - Forward signals
  - Gradient signals

In [7]:
# Deep network simulation with different activation derivative

import numpy as np

def propagate_gradient(depth, derivative):
    g = 1.
    for _ in range(depth):
        g *= derivative
    return g

for d in [10, 20, 50]:
    print(
        d,
        propagate_gradient(d, 0.25),   # sigmoid (best case)
        propagate_gradient(d, 0.8),    # moderate shrinkage
        propagate_gradient(d, 1.0),    # ReLU active
        propagate_gradient(d, 1.1)     # amplification
    )

10 9.5367431640625e-07 0.10737418240000006 1.0 2.5937424601000023
20 9.094947017729282e-13 0.011529215046068483 1.0 6.72749994932561
50 7.888609052210118e-31 1.4272476927059634e-05 1.0 117.39085287969573


### Summary

Think of backpropagation as water flowing upstream through a river system.

- Activation derivatives are the width of each channel
  - Near 0: the channel is almost blocked
  - Near 1: water flows freely
- Weight matrices are pumps or constrictions:
  - Singular values > 1 amplify the flow
  - Singulare values < 1 restrict it
- The entire network is hundreds/thousands of these channels connected in series.

## 9. Signal Propagation Through Deep Networks

> What happens to the signal statistics as it passes through hundreds of random layers?

- Analyse **activation statistics**, not just one layer:
  - Mean
  - Variance
  - Covariance
  - Distribution
- Forward signal propagation
- Gradient signal propagation


> Before worrying about gradients, we must ensure the forward signal itself remains healthy.

- **Xavier initialisation**:
- **He initialisation**:

### How do reasearchers analyse Activation Statistics

```math
z = Wa + b, a' = f'(z)
```

#### Step 1 - Mean propagation

> **Expectation** (E): Long-run average of a random variable.

Suppose expectation of W: $E[W] = 0$. We need to derive $E[z]$.

```math
z_i = \sum_j W_{ij}a_j + b_i \Rightarrow E[z_i] = E[W_{ij}a_j] = E[W_{ij}]E[a_j]
```

Since $E[W] = 0$, and $W_{ij}$ and $a_j$ are independent. So

```math
E[z_i] = 0
```

> Zero-mean weights produce zero-mean pre-activations.

Now, whether post-activations preserve mean or not depends entirely on the activation functions.

- ReLU tends to shift Mean upward, becomes positive

For a zero-mean Gaussian input $Z \sim \mathcal{N}(0, \sigma^2): E[\text{ReLU}(Z) = \frac{\sigma}{\sqrt{2\pi}}]$

- $E[\tanh(z)] = 0$ if z is symmetric around zero
- ELU produces negative ($\alpha(e^x - 1)$) -> Mean remains closer to zero.

#### Step 2 - Variance propagation

Consider:

```math
z_i = \sum_j W_{ij}a_j
```

Fundamental:

- $\text{Var}(X) = E[X^2] - E[X]^2$
- $\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y)$
- $\text{Var}(XY) = E[X^2]E[Y^2] - E[X]^2E[Y]^2$

So

```math
\text{Var}(z_i) = \sum_j \text{Var}(W_{ij}a_j)
```

We have,

```math
\text{Var}(Wa) = E[W^2]E[a^2] - E[W]^2E[a]^2
```

Since $\text{Var}(W) = E[W^2] - E[W]^2 = E[W^2], E[W] = 0$, then

```math
\text{Var}(Wa) = \text{Var}(W)E[a^2]
```

But if $E[a] = 0 \Rightarrow E[a^2] = \text{Var}(a)$. Hence,

```math
\text{Var}(z_i) = n\text{Var}(W)\text{Var}(a)
```

for n := fan-in (number of inputs to the neuron).

##### Xavier initialisation

Now w want the variance stays constant across layers, which means.

```math
\text{Var}(z) = \text{Var}(a)
```

substituded into the equation above,

```math
\text{Var}(a) = n\text{Var}(W)\text{Var}(a)
```

then if $\text{Var}(a)ne 0$,

```math
\text{Var}(W) = \frac{1}{n}
```

Commonly, symmetric derivation that balances both forward and backward:

```math
\text{Var}(W) = \frac{1}{n_{text{in}} + n_{\text{out}}}
```

##### He initialisation

In ReLU, half of inputs are discarded, which means

```math
\text{Var}(\text{ReLU}(Z)) \approx \frac{1}{2}\text{Var}(Z)
```

To compensate, we double the weight variance (substitude into the equation in *step 2*).

```math
n \text{Var}(W) \times \frac{1}{2} = 1 \Leftrightarrow \text{Var}(W) = \frac{2}{n}
```

#### Step 3 - Covariance propagation

> Covariance tells us whether neurons carry redundant information.

For two neurons $a_i, a_j$, the covariance is

```math
\text{Cov}(a_i, a_j) = E[(a_i - \mu_i)(a_j - \mu_j)]
```

- As signals pass through layers, shared weights and nonlinearities change these relationships.
  - High covariance means neurons become similar (redundant).
  - Low covariance means they capture different features.
  - Modern analyses of deep networks often study the evolution of covariance matrices, not just scalar variances, because they reveal how information spreads through the network.

#### Step 4 - Distribution evolution

- **Central limit theorem**, $z_i$ tends to become approx Gaussian at init.
- Activation transforms this Gaussian -> new distribution.


### Implementation

In [8]:
import numpy as np

np.random.seed(42)

width = 1000
depth = 20

x = np.random.randn(width)

for layer in range(depth):
    W = np.random.randn(width, width) * np.sqrt(2 / width)
    x = W @ x
    x = np.maximum(0, x)

    print(
        f"Layer {layer+1:2d} | "
        f"Mean = {x.mean():8.4f} | "
        f"Variance = {x.var():8.4f}"
    )

Layer  1 | Mean =   0.5242 | Variance =   0.6274
Layer  2 | Mean =   0.5588 | Variance =   0.6300
Layer  3 | Mean =   0.5577 | Variance =   0.6512
Layer  4 | Mean =   0.5528 | Variance =   0.6778
Layer  5 | Mean =   0.5473 | Variance =   0.6465
Layer  6 | Mean =   0.5130 | Variance =   0.5994
Layer  7 | Mean =   0.5476 | Variance =   0.5821
Layer  8 | Mean =   0.5520 | Variance =   0.6373
Layer  9 | Mean =   0.4972 | Variance =   0.5541
Layer 10 | Mean =   0.5184 | Variance =   0.5402
Layer 11 | Mean =   0.5196 | Variance =   0.6134
Layer 12 | Mean =   0.5491 | Variance =   0.6567
Layer 13 | Mean =   0.5694 | Variance =   0.6873
Layer 14 | Mean =   0.5879 | Variance =   0.7325
Layer 15 | Mean =   0.5592 | Variance =   0.6694
Layer 16 | Mean =   0.5909 | Variance =   0.6937
Layer 17 | Mean =   0.5446 | Variance =   0.6630
Layer 18 | Mean =   0.5279 | Variance =   0.6001
Layer 19 | Mean =   0.5395 | Variance =   0.6177
Layer 20 | Mean =   0.5105 | Variance =   0.5618


## 10. Activation Functions and Function Approximation

> Why do activation functions make neural networks intelligent?
> Why does a tiny non-linearity makes a network represent data?

No activation functions:

> **Linear Collapse Theorem**: Linear network = single linear layer.

- Many Linears = Linear regions.
- local linears -> global non-linears

### Universal Approximation Theorem

> A feedforward neural network with:
> - One hidden layer,
> - A non-linear activation satisfying mild conditions,
> - Sufficiently many hidden neurons.
> 
> can approximate any continuous function on a compact domain arbitrarily well.

## 11. Activation Functions in Convolutional Networks

- Input: $X \in \mathbb{R}^{C \times H \times W}$
- Huge amount of params (pixels) each layer -> Cheap activation -> ReLU.
- Some other activations are *slightly* better, but the computational cost is high.

For $X \in \mathbb{R}^{C \times H \times W}$, activation is

```math
Y_{c, h, w} = \max(0, X_{c, h, w})
```

- **Sparsity**: $P(X > 0) = 0.5$ -> half the features disappear completely -> skipped -> efficiency increases.

> CNN block = Conv > Batch norm > ReLU.

**Batch norm**

```math
\^{z} = \frac{z - \mu}{\sigma} \rightarrow y = \gamma \^{z} + \beta
```

- Distribution centred around zero. -> ReLU keeps roughly 50% alive.
- Batch norm scales hugh mean and variance (e.g., $\mu = 20, \sigma = 300$) to: $\mu = 0, \sigma = 1$.

**However**

- Modern picture:
  - **Classical CNNs** (AlexNet, VGG, ResNet, DenseNet, EfficientNet): predominantly ReLU.
  - **Transformer-inspired CNNs** (ConvNeXt and similar): often GELU.
- Because:
  - GPUs became much faster at evaluating transcendental functions.
  - Frameworks fuse operations into highly optimised kernels.
  - The cost of convolutions decreased relative to other parts of the network.
  - Accuracy improvements of smooth activations became worthwhile for very large models.

## 12. Activation Functions in Transformers

- Transformer is more complex, deeper. It needs smoothness, continuous output.
- Binary gate (on, off) - ReLU -> How strong a feature contributes - GELU.
- They even add another neuron to decide how much info flows, turning fixed gate -> learnable gate.
  - GLU
  - GEGLU
  - SwiGLU
- Withou these gates, each feature is treated independently.

### GELU

Gaussian Cumulative Distribution function

```math
\Phi(x) = \int_{-\infin}^x \frac{1}{\sqrt{2\pi}} e^{-t^2 / 2} dt
```

then

```math
P(\text{keep}) = \Phi(x)
```

and the activation function becomes:

```math
f(x) = x\Phi(x)
```

- GELU is smooth everywhere.
- Negative values aren't discarded. -> down-weighted.

### GLU (Gated Linear Units)

- Fixed gate -> learned gate.
- Use Sigmoid to learn gate.
- Keep everything equally.

For input $x$, $a = xW_a, b = xW_b$, then:

```math
\text{GLU}(x) = a \odot \sigma(b)
```

In implementation, they use:

```math
0.5x(1 + \tanh(\sqrt{\frac{2}{\pi}}(x + 0.044715x^3)))
```

### GEGLU (GELU Gated Linear Units)

- Replace sigmoid = GELU

```math
\text{GEGLU}(x) = a \odot \text{GELU}(b)
```

### SwiGLU

- Replace sigmoid with Swish/SiLU

```math
\text{SwiGLU} = a \odot \text{SiLU}(b)
```

### In PyTorch

In [9]:
import numpy as np

def relu(x):
    return np.maximum(0, x)

def gelu(x):
    return 0.5 * x * (
        1 + np.tanh(
            np.sqrt(2 / np.pi) *
            (x + 0.044715 * x**3)
        )
    )

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def silu(x):
    return x * sigmoid(x)

def glu(a, b):
    return a * sigmoid(b)

def geglu(a, b):
    return a * gelu(b)

def swiglu(a, b):
    return a * silu(b)

In [11]:
import torch
import torch.nn.functional as F

x = torch.randn(100)
np_out = gelu(x.numpy())
torch_out = F.gelu(x).numpy()

print(np.max(np.abs(np_out - torch_out)))

AttributeError: partially initialized module 'torch' has no attribute 'distributed' (most likely due to a circular import)

## 13. Numerical Stability

> This lesson bridges mathematics and computer systems.

- A neuron doesn't evaluate a function, it *approximate* a math function under *finite memory*.

### Stable Sigmoid

```math
\sigma(x) = \frac{1}{1 + e^{-x}}
```

if $x \gg 0$, $\sigma(x) = 0$ -> OK.

But if $x \ll 0$ -> $\sigma(x)$ is OVERFLOW! -> Bad!

To solve the latter case, multiply both numerator and denominator by $e^x$

```math
\sigma(x) = \frac{e^x}{1 + e^x}
```

This time it is easier to compute. It underflows to 0 instead of overflowing.

```math
\sigma(x) = \begin{cases}
    \frac{1}{1 + e^{-x}}, & x \ge 0 \\
    \frac{e^x}{1 + e^x}, & x < 0
\end{cases}
```

In [13]:
import numpy as np

def stable_sigmoid(x):
    x = np.asarray(x)
    out = np.empty_like(x, dtype=np.float64)

    positive = x >= 0
    negative = ~positive

    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))

    exp_x = np.exp(x[negative])
    out[negative] = exp_x / (1 + exp_x)

    return out

def naive_sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [14]:
x = np.array([-1000, 0, 1000])

print(stable_sigmoid(x))
print(naive_sigmoid(x))

[0.  0.5 1. ]
[0.  0.5 1. ]


C:\Users\PC\AppData\Local\Temp\ipykernel_35988\3602117641.py:18: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


In [17]:
import torch
import torch.nn.functional as F

x = torch.linspace(-100, 100, 1000)

torch_out = torch.sigmoid(x).numpy()
numpy_out = stable_sigmoid(x.numpy())

print(np.max(np.abs(torch_out - numpy_out)))

AttributeError: partially initialized module 'torch' has no attribute 'distributed' (most likely due to a circular import)

### Stable Softplus

```math
\text{Softplus}(x) = \log(1 + e^x)
```

Using similar approach

```math
\text{Softplus}(x) = \begin{cases}
    \log(1 + e^x), & x \le 0 \\
    x + log(1 + e^{-x}), & x > 0
\end{cases}
```

In [15]:
def stable_softplus(x):
    x = np.asarray(x)

    return np.where(
        x > 0,
        x + np.log1p(np.exp(-x)),
        np.log1p(np.exp(x))
    )

### Floating point precision

```math
0.1 + 0.2 = 0.30000000000000004
```

because computer cannot encode 0.1 exactly.

- Modern computers use float16 instead of float32 -> sooner overflow -> **loss scaling**.
- Many GPU kernels use:
  - polynomial approximations
  - fused operations
  - branch-minimising implementations

In [16]:
x32 = np.array([1.234567], dtype=np.float32)
x16 = np.array([1.234567], dtype=np.float16)

print(x32)
print(x16)

[1.234567]
[1.234]


## 14. Implementing an Activation Library

In [20]:
import os, sys
sys.path.append("../../")
from src import ReLU, Sigmoid, Tanh, GELU, Swish, Mish

ACTIVATIONS = {
    "relu": ReLU,
    "sigmoid": Sigmoid,
    "tanh": Tanh,
    "gelu": GELU,
    "swish": Swish,
    "mish": Mish
}

def get_activation(name):
    return ACTIVATIONS[name.lower()]()

In [21]:
activation = get_activation("gelu")
y = activation(x)

In [22]:
import torch
import torch.nn as nn

x = torch.randn(10)
torch_relu = nn.ReLU()(x)
numpy_relu = ReLU()(x.numpy())

print(np.allclose(torch_relu.numpy(), numpy_relu))

AttributeError: partially initialized module 'torch' has no attribute 'distributed' (most likely due to a circular import)

## 15. Research Perspective: Nonlinearity as Computation